# IDEIAS INICIAIS E EXTRAÇÃO DOS DADOS

## O Conceito Central - Utilizar um Modelo Híbrido

A ideia do modelo híbrido é unir "o melhor dos dois mundos": o poder das Redes Neurais para encontrar padrões complexos e ocultos nos dados, combinado com a transparência da Estatística Clássica para tomar e justificar a decisão final de investimento. Ele funciona em duas etapas principais:

Etapa 1: O Analista (A Rede Neural - MLP)

O que faz: Um Multi-Layer Perceptron (MLP) "raso" processa diversos indicadores de mercado (como preços históricos, volume, volatilidade e, se quiser, dados de balanço).

O Objetivo: A rede neural não toma a decisão de compra ou venda. Ela apenas encontra relações complexas e não-lineares nesses dados e cospe um único número: um "Fator de Risco" (ou Score) para cada ação.

Vantagem: Sendo "rasa" (poucas camadas), ela treina muito rápido, permitindo rodar o backtest de vários anos em segundos.

Etapa 2: O Gestor (O Modelo Linear - GLM)

O que faz: O Modelo Linear Generalizado (GLM) recebe o "Fator de Risco" gerado pelo MLP e o coloca lado a lado com preditores clássicos (como o momentum simples da ação).

O Objetivo: O GLM calcula a previsão final (ex: o retorno esperado para o próximo mês) e gera o ranking que dirá ao robô quais são as 10 ações para comprar e as 10 para vender.

Por que essa estratégia é brilhante para o Desafio? A banca avaliadora costuma penalizar modelos de Machine Learning puramente "caixa-preta" (onde a máquina compra a ação e você não sabe explicar o porquê). Com o modelo híbrido, você resolve isso. Como a decisão final é tomada pelo GLM, você tem acesso aos coeficientes matemáticos ($\beta$) de cada variável. 

Objetivo: 

Nosso robô utilizar uma rede neural para ler o mercado, mas a decisão final ser totalmente explicável. Conseguir provar estatisticamente que o Fator de Risco gerado pela IA teve um peso de X% na escolha das ações, mitigando vieses.

### Estratégia quantitativa multifatorial long-short com modelo híbrido de Machine Learning e Estatística.

Nossa estratégia não utiliza a rede neural para decidir diretamente quais ações comprar. Em vez disso, ela utiliza o MLP como um extrator de fatores (feature extractor), capaz de capturar relações não lineares entre variáveis de mercado. O fator produzido pela rede neural é então incorporado a um modelo estatístico explicável (GLM), juntamente com fatores tradicionais, como momentum, volatilidade e liquidez. A decisão final da carteira long-short é tomada pelo GLM, permitindo interpretar a contribuição estatística de cada fator por meio de seus coeficientes.

### Como isso funcionaria?

Para organizar os nossos dados de 2020 a 2026 mantendo o rigor metodológico exigido para o backtest, devemos usar o período de 2020 a 2022 exclusivamente para treinar a Rede Neural (MLP), fazendo com que ela aprenda os padrões históricos não-lineares; em seguida, aplique essa rede para gerar o Score das ações na janela de 2023 a 2024, utilizando esses dois anos para treinar o Modelo Linear (GLM), que aprenderá a ponderar as falhas e acertos da IA contra os indicadores clássicos (como volatilidade); por fim, reserve o período de 2025 até 2026 para a Operação Real (Backtest), simulando os investimentos do nosso robô em um cenário de mercado que nenhum dos dois modelos jamais viu, o que garantirá uma prova à prova de balas contra o sobreajuste (overfitting) para apresentar à banca avaliadora.

## Extração dos Dados Utilizados pelo Modelo

In [1]:
pip install yfinance pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 1.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [yfinance]/12 [yfinance]]oup4]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import yfinance as yf
import pandas as pd
import numpy as np

def preparar_dados_quant():
    # 1. Definir o universo de ativos
    tickers = [
        'VALE3.SA', 'PETR4.SA', 'ITUB4.SA', 'BBDC4.SA', 'B3SA3.SA', 
        'ABEV3.SA', 'WEGE3.SA', 'RADL3.SA', 'RENT3.SA', 'BBAS3.SA'
    ]
    
    print("Baixando dados do Yahoo Finance...")
    df_raw = yf.download(tickers, start='2020-01-01', end='2026-07-06', auto_adjust=True)['Close']

    # Se algum ticker falhar e vier totalmente vazio (NaN), ele é removido aqui
    df_raw.dropna(axis=1, how='all', inplace=True)
    
    # 2. Engenharia de Features (Vetorizada para máxima eficiência e escalabilidade)
    print("Calculando features explicáveis...")
    
    daily_log_returns = np.log(df_raw / df_raw.shift(1))
    momentum_6m = np.log(df_raw / df_raw.shift(126))
    volatility_6m = daily_log_returns.rolling(window=126).std() * np.sqrt(252)
    risk_adj_momentum = momentum_6m / volatility_6m
    
    # 3. Variável Resposta (Target Y)
    target_forward_1m = (df_raw.shift(-21) / df_raw) - 1
    
    print("Formatando a matriz de dados...")
    
    df_features = pd.DataFrame({
        'preco_fechamento': df_raw.unstack(),
        'momentum_6m': momentum_6m.unstack(),
        'volatilidade_6m': volatility_6m.unstack(),
        'momentum_ajustado': risk_adj_momentum.unstack(),
        'target_retorno_1m': target_forward_1m.unstack()
    }).reset_index()
    
    # Forçar o nome das colunas diretamente pela posição
    df_features.columns = ['ticker', 'data', 'preco_fechamento', 'momentum_6m', 
                           'volatilidade_6m', 'momentum_ajustado', 'target_retorno_1m']
    
    # Remover dias com dados faltantes (período de lookback)
    df_features.dropna(inplace=True)
    
    # Ordenar por data para evitar vazamento de dados no backtest
    df_features.sort_values(by=['data', 'ticker'], inplace=True)
    df_features.reset_index(drop=True, inplace=True)
    
    print("Dados prontos para modelagem!")
    return df_features

# Executar a função
df_preparado = preparar_dados_quant()

# Visualizar as primeiras linhas do dataset
print(df_preparado.head())

[**********************70%*********              ]  7 of 10 completed

Baixando dados do Yahoo Finance...


[*********************100%***********************]  10 of 10 completed

Calculando features explicáveis...
Formatando a matriz de dados...
Dados prontos para modelagem!
     ticker       data  preco_fechamento  momentum_6m  volatilidade_6m  \
0  ABEV3.SA 2020-07-06         10.732965    -0.290464         0.548003   
1  B3SA3.SA 2020-07-06         14.846695     0.244531         0.698607   
2  BBAS3.SA 2020-07-06         11.119793    -0.425416         0.816154   
3  BBDC4.SA 2020-07-06         12.411107    -0.401609         0.712554   
4  ITUB4.SA 2020-07-06         17.317348    -0.282329         0.584328   

   momentum_ajustado  target_retorno_1m  
0          -0.530040          -0.066852  
1           0.350026           0.118542  
2          -0.521246          -0.039515  
3          -0.563620          -0.040239  
4          -0.483168          -0.073050  


### Colunas

1. ticker (O Identificador)O que é: O código da ação na bolsa (ex: VALE3.SA, PETR4.SA).Para que serve: Identifica a qual empresa pertencem os dados daquela linha. No backtest, o seu robô usará o ticker para saber qual ativo ele deve colocar na lista de compra (Long) ou de venda (Short).

2. data (A Linha do Tempo)O que é: O dia exato em que aqueles indicadores foram calculados.Para que serve: É crucial para o backtest. O robô vai agrupar os dados por data (ex: pegar tudo do último dia do mês) para tomar a decisão sem "espiar o futuro" (data leakage).

3. preco_fechamento (O Dado Bruto)O que é: O preço da ação no fim daquele dia. Como usamos auto_adjust=True, este preço já tem os dividendos e desdobramentos descontados.Para que serve: Garante que o retorno calculado seja real. Se uma ação custa R$ 20 e paga R$ 2 de dividendos, o preço ajustado impede que o robô ache que a ação "caiu" e a venda por engano.

4. momentum_6m (Preditor / Matriz X)O que é: O retorno logarítmico acumulado da ação olhando para trás (últimos 126 dias úteis / ~6 meses).Para que serve: É a medida clássica de tendência. Ele diz ao modelo: "Nos últimos 6 meses, esta ação subiu X%". Será um dos inputs principais para a sua Rede Neural (MLP).

5. volatilidade_6m (Preditor / Matriz X)O que é: O risco da ação. O desvio padrão anualizado dos retornos diários ao longo dos mesmos 6 meses.Para que serve: Diz ao modelo o quão "nervosa" a ação está. O seu GLM poderá usar esta coluna para penalizar ativos que subiram, mas que oscilam de forma violenta.

6. momentum_ajustado (Preditor / Matriz X)O que é: A coluna momentum_6m dividida pela volatilidade_6m.Para que serve: É o nosso "filtro de incerteza". Um valor alto aqui significa que a ação subiu de forma suave e consistente. Um valor baixo ou negativo significa que ela andou de lado ou caiu de forma volátil. É uma feature excelente para dar previsibilidade ao modelo híbrido.

7. target_retorno_1m (O Gabarito / Variável Y)O que é: O retorno percentual real que a ação teve 21 dias úteis depois daquela data (aproximadamente 1 mês no futuro).Para que serve: Esta é a coluna mais importante para o treino da IA! É a variável resposta ($Y$). Durante o treino, o seu modelo vai olhar para as colunas 4, 5 e 6 (o passado/presente) e tentar prever a coluna 7 (o futuro).Como a mágica acontece na prática:Quando você for rodar a simulação para o mês que vem, a coluna target_retorno_1m estará vazia (afinal, não sabemos o futuro). O seu modelo treinado vai cruzar o momentum e a volatilidade daquele dia, calcular uma previsão numérica para preencher esse "buraco" e montar o ranking: as 10 ações com a previsão mais alta ele compra, as 10 com a menor ele vende!

In [ ]:
#Salva os dados em .csv
df_preparado.to_csv("dados.csv", index=False)